# AML validation and preliminary gap analysis

Read-only companion to Phase 3. It loads the saved validation predictions for Logistic Regression and XGBoost, checks that they cover the same flights, and plots ranking, calibration, and threshold behavior. **This is not the final Phase 5 evaluation.** No held-out test outcomes are read, and no operational precision floor or threshold is chosen here. Run `python -m src.models.classical.train` first if artifacts are missing.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/project_contract.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open JupyterLab from the repository root (or its notebooks folder).")
print("Repository:", ROOT)
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve, roc_curve
from sklearn.calibration import calibration_curve
BASE = ROOT / "artifacts/classical"
META = json.loads((BASE / "run_metadata.json").read_text())
PRED = {
    name: pd.read_csv(BASE / filename)
    for name, filename in {
        "Logistic Regression": "logistic_validation_predictions.csv",
        "XGBoost": "xgboost_validation_predictions.csv",
    }.items()
}
for name, frame in PRED.items():
    assert frame.flight_row_id.is_unique and frame.split.eq("validation").all()
    assert frame.y_probability.between(0,1).all() and np.isfinite(frame.y_probability).all()
    print(name, len(frame), "validation predictions")
left, right = PRED.values()
paired = left[["flight_row_id","y_true"]].merge(right[["flight_row_id","y_true"]],on="flight_row_id",validate="one_to_one",suffixes=("_lr","_xgb"))
assert len(paired) == len(left) == len(right) and paired.y_true_lr.equals(paired.y_true_xgb)
print("Shared validation prevalence:", round(left.y_true.mean(),4))


## Same-population validation metrics

PR-AUC is average precision. Brier score assesses probability error; a reliability plot below shows how that error varies across prediction bins.

In [ ]:
rows=[]
for name, frame in PRED.items():
    y, p = frame.y_true.to_numpy(), frame.y_probability.to_numpy()
    rows.append({"model":name,"n":len(frame),"prevalence":y.mean(),"pr_auc":average_precision_score(y,p),"roc_auc":roc_auc_score(y,p),"brier_score":brier_score_loss(y,p)})
metrics=pd.DataFrame(rows).set_index("model")
print(metrics.round(4).to_string())


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for name, frame in PRED.items():
    y,p=frame.y_true.to_numpy(),frame.y_probability.to_numpy()
    precision,recall,_=precision_recall_curve(y,p)
    fpr,tpr,_=roc_curve(y,p)
    axes[0].plot(recall,precision,label=name)
    axes[1].plot(fpr,tpr,label=name)
axes[0].axhline(left.y_true.mean(),color="gray",linestyle="--",label="Validation prevalence")
axes[0].set(xlabel="Recall",ylabel="Precision",title="Validation precision–recall")
axes[1].plot([0,1],[0,1],color="gray",linestyle="--")
axes[1].set(xlabel="False positive rate",ylabel="True positive rate",title="Validation ROC")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()


## Probability calibration

Quantile bins each contain roughly equal numbers of validation flights. These plots are diagnostics, not fitted calibrators.

In [ ]:
fig, ax=plt.subplots(figsize=(6,5))
for name, frame in PRED.items():
    observed, predicted=calibration_curve(frame.y_true,frame.y_probability,n_bins=10,strategy="quantile")
    ax.plot(predicted,observed,marker="o",label=name)
ax.plot([0,1],[0,1],color="gray",linestyle="--",label="Perfect calibration")
ax.set(xlabel="Mean predicted probability",ylabel="Observed delay frequency",title="Validation reliability (10 quantile bins)")
ax.legend(); plt.tight_layout(); plt.show()


## Precision and recall behavior

This curve helps choose a precision requirement later. It does **not** choose a threshold: the project contract labels 0.70 as an unapproved candidate. The operational rule, calibration decision, and final test evaluation belong in Phase 5.

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
for name,frame in PRED.items():
    precision,recall,thresholds=precision_recall_curve(frame.y_true,frame.y_probability)
    ax.plot(precision[:-1],recall[:-1],label=name,alpha=.8)
ax.set(xlabel="Achieved validation precision",ylabel="Validation recall",title="Alert trade-off across thresholds",xlim=(0,1),ylim=(0,1))
ax.legend(); plt.tight_layout(); plt.show()


## Identify the Gap — preliminary, validation evidence only

On the saved validation period, Logistic Regression has higher PR-AUC (0.3189 versus 0.2945) and ROC-AUC (0.6815 versus 0.6745), while XGBoost has a lower Brier score (0.1247 versus 0.1333). This is a measured ranking versus probability-error trade-off. The plots above show where probabilities diverge from observed frequencies and how recall falls as precision requirements rise. The validation period has a different delay prevalence from training and test, so threshold behavior may shift. Weather is retrospective, and historical outcomes still need an availability check at T−60. These limitations motivate further investigation; they do not demonstrate that deep learning will improve performance.

**Remaining for the current AML aim:** resolve historical feature timing; agree or clearly label an exploratory precision floor; select thresholds on validation data; decide whether to fit calibration using only train/validation data; freeze decisions; evaluate both models once on the same held-out test population; then update the gap discussion with test evidence. This is Phase 5 AML work. Phase 4 remains DL.